# Importaciones

## Descarga del corpus

In [ ]:
# !gdown 1SXH1pwWV1-M32yyJ8PUYGi5W3N1G3oBz

Downloading...
From: https://drive.google.com/uc?id=1SXH1pwWV1-M32yyJ8PUYGi5W3N1G3oBz
To: /home/Fer/Repos/TareasUnison/2026-1/PLN/Actividades/Actividad-13/all_top_rated_movies_from_TMDB.csv
100%|██████████████████████████████████████| 3.24M/3.24M [00:01<00:00, 2.72MB/s]


In [3]:
import pandas as pd

df = pd.read_csv('all_top_rated_movies_from_TMDB.csv')
df = df[['original_title','overview']].copy()
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
docs = df['overview'].to_list()
df

,original_title,overview
0,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...
1,The Godfather,"Spanning the years 1945 to 1955, a chronicle o..."
2,The Godfather Part II,In the continuing saga of the Corleone crime f...
3,Schindler's List,The true story of how businessman Oskar Schind...
4,12 Angry Men,The defense and the prosecution have rested an...
...,...,...
9774,Vehicle 19,A parolee becomes the target of a massive poli...
9775,Contracted,A young woman contracts what she believes to b...
9776,Dos,"Two people, a man and a woman, wake up naked a..."
9777,Teeth,Dawn is an active member of her high-school ch...


___

# Visualización de docs

In [18]:
import random

for n in range(3):
    al = random.randint(0, len(docs)-1)
    print(f'{'-' * 300}\n')
    print(f'{al}: {docs[al]}')
    print(f'\n{'-' * 300}')


------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

584: Chan Wing Yan, a young police officer, has been sent undercover as a mole in the local mafia. Lau Kin Ming, a young mafia member, infiltrates the police force. Years later, their older counterparts, Chen Wing Yan and Inspector Lau Kin Ming, respectively, race against time to expose the mole within their midst.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------

___

# Modelo

In [14]:
from sentence_transformers import SentenceTransformer

model_name = "all-mpnet-base-v2"

model = SentenceTransformer(model_name)

/home/Fer/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14329.53it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
embeddings = model.encode(docs)
embeddings.shape

(9779, 768)

___

# Querys

## Textos excogidos

In [20]:
print(docs[584])

Chan Wing Yan, a young police officer, has been sent undercover as a mole in the local mafia. Lau Kin Ming, a young mafia member, infiltrates the police force. Years later, their older counterparts, Chen Wing Yan and Inspector Lau Kin Ming, respectively, race against time to expose the mole within their midst.


In [21]:
print(docs[1])

Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.


In [100]:
query584_textual = "Chan Wing Yan goes undercover to infiltrate the local mafia"
query584_parafrasis = "Amidst a prolonged deception, two moles swap roles between law enforcement and triads. Eventually, these seasoned doubles scramble to unmask each other before their true identities surface."

query1_textual = "Michael Corleone launches a campaign of bloody revenge"
query1_parafrasis = "Over a decade post-WWII, this tale follows an immigrant syndicate. After the chieftain survives an ambush, his junior offspring executes the culprits, triggering lethal retribution"

In [101]:
query584_textual_vector = model.encode([query584_textual])
query584_parafrasis_vector = model.encode([query584_parafrasis])

query1_textual_vector = model.encode([query1_textual])
query1_parafrasis_vector = model.encode([query1_parafrasis])

In [102]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=10, metric='cosine')

nn.fit(embeddings)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [103]:
neighbors584_textual = nn.kneighbors(query584_textual_vector)
neighbors584_parafrasis = nn.kneighbors(query584_parafrasis_vector)
neighbors1_textual = nn.kneighbors(query1_textual_vector)
neighbors1_parafrasis = nn.kneighbors(query1_parafrasis_vector)

In [104]:
distancias, idxs = neighbors584_textual
for idx in idxs[0]:
    print(docs[idx])
    print(50*"-")

Chan Wing Yan, a young police officer, has been sent undercover as a mole in the local mafia. Lau Kin Ming, a young mafia member, infiltrates the police force. Years later, their older counterparts, Chen Wing Yan and Inspector Lau Kin Ming, respectively, race against time to expose the mole within their midst.
--------------------------------------------------
Hong Kong cop Chan Ka-Kui returns, working with Interpol to track down and arrest an illegal weapons dealer. Chan later realizes that things are not as simple as they appear and soon finds himself to be a pawn of an organization posing as Russian intelligence.
--------------------------------------------------
Sent into a drunken tailspin when his entire unit is killed by a gang of thrill-seeking punks, disgraced Hong Kong police inspector Wing needs help from his new rookie partner, with a troubled past of his own, to climb out of the bottle and track down the gang and its ruthless leader.
---------------------------------------

In [105]:
distancias, idxs = neighbors584_parafrasis
for idx in idxs[0]:
    print(docs[idx])
    print(50*"-")

To take down South Boston's Irish Mafia, the police send in one of their own to infiltrate the underworld, not realizing the syndicate has done likewise. While an undercover cop curries favor with the mob kingpin, a career criminal rises through the police ranks. But both sides soon discover there's a mole among them.
--------------------------------------------------
Chan Wing Yan, a young police officer, has been sent undercover as a mole in the local mafia. Lau Kin Ming, a young mafia member, infiltrates the police force. Years later, their older counterparts, Chen Wing Yan and Inspector Lau Kin Ming, respectively, race against time to expose the mole within their midst.
--------------------------------------------------
A DEA agent and an undercover Naval Intelligence officer who have been tasked with investigating one another find they have been set up by the mob -- the very organization the two men believe they have been stealing money from.
--------------------------------------

In [106]:
distancias, idxs = neighbors1_textual
for idx in idxs[0]:
    print(docs[idx])
    print(50*"-")

In the midst of trying to legitimize his business dealings in 1979 New York and Italy, aging mafia don, Michael Corleone seeks forgiveness for his sins while taking a young protege under his wing.
--------------------------------------------------
Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.
--------------------------------------------------
After being set-up and betrayed by the man who hired him to assassinate a Texas Senator, an ex-Federale launches a brutal rampage of revenge against his former boss.
--------------------------------------------------
A former mob enforcer who is released from prison after serving 22 years for a crime he didn't commit sets out on a path for revenge against the people who wronged him.
-------

In [107]:
distancias, idxs = neighbors1_parafrasis
for idx in idxs[0]:
    print(docs[idx])
    print(50*"-")

A simple Chinese immigrant wages a perilous war against one of the most powerful criminal organizations on the planet.
--------------------------------------------------
In the post–World War II South, two families are pitted against a barbaric social hierarchy and an unrelenting landscape as they simultaneously fight the battle at home and the battle abroad.
--------------------------------------------------
In their new overseas home, an American family soon finds themselves caught in the middle of a coup, and they frantically look for a safe escape in an environment where foreigners are being immediately executed.
--------------------------------------------------
When a hostage rescue mission creates a new enemy, Capt. Guerrero and his elite soldiers must face an ambush by a criminal group.
--------------------------------------------------
A group of Mexican emigrants attempts to cross the Mexican-US border. What begins as a hopeful journey becomes a harrowing, bloody and primal f

¿Encontraste, con ambas queries cada uno de los dos textos de referencia? Rellena la información abajo.
| Documento de referencia 1 |
| Query textual:    Fue encontrado en la posición 1
| Query paráfrasis: Fue encontrado en la posición 2 

| Documento de referencia 2 |
| Query textual:    Fue encontrado en la posición 2
| Query paráfrasis: No fue encontrado en los primeros 10 

________________________________________________________________________________

¿Con cuál de las dos queries fué mejor?
R/: En ambos casos fue mejor con la textual

________________________________________________________________________________

¿Qué puedes concluir sobre el desempeño de estos embeddings? Es decir, ¿qué información del texto captura mejor un modelo basado en transformadores? ¿la información textual o semántica?
R/: La información textual/